In [1]:
import pyautogui          #more library installation
import pytesseract
import cv2
import numpy as np
import pandas as pd
import time
import math
import sys
import re
import statistics
import os
import subprocess
import requests


In [2]:
#Installs pytesseract for scanning
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"


In [3]:
pyautogui.FAILSAFE = True;   #security mechanism 

In [ ]:
#def locate_coordinates():     #function to find coordinates on computer screen
#    print("Move your mouse to see coordinates. Press Ctrl+C to stop.\n")
#    try:
#        while True:
#            # Get current mouse position
#            x, y = pyautogui.position()
            # Print coordinates in a fixed-width format
#            print(f"X: {x:4}  Y: {y:4}", end="\r", flush=True)
#            time.sleep(0.05)  # Update every 50ms
#    except KeyboardInterrupt:
#        print("\nStopped tracking.")
#        sys.exit(0)
#    except Exception as e:
#        print(f"\nError: {e}")
#        sys.exit(1)

#if __name__ == "__main__":
#    locate_coordinates()

In [4]:
# Load coordinates of buttons
xcoor = dict(activate_rocker = 81, left_rocker = 540, right_rocker = 1440, unlock_lift = 73 , lift_off = 74, take_picture = 77, toggle_speed=78, lift_slide1 = 777, lift_slide2 = 1148);
ycoor = dict(activate_rocker = 522, left_rocker = 579, right_rocker = 576 , unlock_lift = 985 , lift_off = 826, take_picture = 239, toggle_speed=674, lift_slide1 = 592, lift_slide2 = 585);
xtemp = dict(r1_up = 536, r1_down = 538, r1_left = 362, r1_right = 713);
ytemp = dict(r1_up = 386, r1_down = 763, r1_left = 576, r1_right = 565);
xcoor.update(xtemp);
ycoor.update(ytemp);
xtemp = dict(r2_up = 1438, r2_down = 1435, r2_left = 1251, r2_right = 1620);
ytemp = dict(r2_up = 384, r2_down = 760, r2_left = 572, r2_right = 569);
xcoor.update(xtemp);
ycoor.update(ytemp);

In [5]:
# Initiate flags
rockers_activated = False;
in_flight = False;

In [6]:
# Define commands
def activate_rockers():
    global rockers_activated;
    if(not rockers_activated):
        pyautogui.click(xcoor['activate_rocker'], ycoor['activate_rocker']);
        rockers_activated = True;
def deactivate_rockers():
    global rockers_activated;
    if(rockers_activated):
        pyautogui.click(xcoor['activate_rocker'], ycoor['activate_rocker']);
        rockers_activated = False;
def move_drone(direction,duration):
    if(not rockers_activated):
        activate_rockers();
    if(direction == 'forward'):
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
        pyautogui.mouseDown(xcoor['right_rocker'], ycoor['right_rocker'], button='left')
        pyautogui.moveTo(xcoor['r2_up'], ycoor['r2_up'], duration=0.5);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r2_up'], ycoor['r2_up'], button='left')
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
    elif(direction == 'backward'):
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
        pyautogui.mouseDown(xcoor['right_rocker'], ycoor['right_rocker'], button='left')
        pyautogui.moveTo(xcoor['r2_down'], ycoor['r2_down'], duration=0.5);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r2_down'], ycoor['r2_down'], button='left')
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
    elif(direction == 'left'):
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
        pyautogui.mouseDown(xcoor['right_rocker'], ycoor['right_rocker'], button='left')
        pyautogui.moveTo(xcoor['r2_left'], ycoor['r2_left'], duration=0.5);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r2_left'], ycoor['r2_left'], button='left')
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
    elif(direction == 'right'):
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
        pyautogui.mouseDown(xcoor['right_rocker'], ycoor['right_rocker'], button='left')
        pyautogui.moveTo(xcoor['r2_right'], ycoor['r2_right'], duration=0.5);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r2_right'], ycoor['r2_right'], button='left')
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
    elif(direction == 'up'):
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
        pyautogui.mouseDown(xcoor['left_rocker'], ycoor['left_rocker'], button='left')
        pyautogui.moveTo(xcoor['r1_up'], ycoor['r2_up'], duration=0.5);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r1_up'], ycoor['r1_up'], button='left')
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
    elif(direction == 'down'):
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
        pyautogui.mouseDown(xcoor['left_rocker'], ycoor['left_rocker'], button='left')
        pyautogui.moveTo(xcoor['r1_down'], ycoor['r1_down'], duration=0.5);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r1_down'], ycoor['r1_down'], button='left')
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
def turn_drone(direction,duration):
    if(not rockers_activated):
        activate_rockers();
    if(direction == 'left'):
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
        pyautogui.mouseDown(xcoor['left_rocker'], ycoor['left_rocker'], button='left')
        pyautogui.moveTo(xcoor['r1_left'], ycoor['r1_left'], duration=0.5);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r1_left'], ycoor['r1_left'], button='left')
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
    elif(direction == 'right'):
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
        pyautogui.mouseDown(xcoor['left_rocker'], ycoor['left_rocker'], button='left')
        pyautogui.moveTo(xcoor['r1_right'], ycoor['r1_right'], duration=0.5);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r1_right'], ycoor['r1_right'], button='left')
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);

In [7]:

def clean_num(s: str) -> str:    #function receives a string and outputs a string 
    s = s.replace(',', '.')     #replace comma with period
    s = re.sub(r'[^0-9\.\-]', '', s)  #redefines inout string to replace anything not a neg sign, number, or dec point with space
    return s

In [8]:
# Function to capture a specific region using bounding box coordinates (left, top, width, height)

def read_coords():
    region = (920, 35, 150, 80)   
    region_screenshot = pyautogui.screenshot(region=region)
    img = cv2.cvtColor(np.array(region_screenshot), cv2.COLOR_BGR2GRAY)

    _, processed_img = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

    text = pytesseract.image_to_string(processed_img)
    text = text.replace('o','0').replace('O','0')
    lines = text.splitlines()
    
    #print(repr(text))
    #print(lines)

    try:   # this prevents error values from being accepted
        latitude_read = float(clean_num(lines[1]))
        longitude_read = float(clean_num(lines[0]))
        
        if 'W' in lines[0] and longitude_read>0:
            longitude_read = -longitude_read

        return latitude_read, longitude_read

    except (IndexError, ValueError):
        return None, None


In [9]:
#averages the coordiantes found for better control

def read_coords_stable(n=3, delay=0.04):
    lats, lons = [], []

    for _ in range(n):
        lat, lon = read_coords()
        if lat is not None and lon is not None:
            lats.append(lat)
            lons.append(lon)
        time.sleep(delay)

    if not lats:
        return None, None

    return statistics.median(lats), statistics.median(lons)


In [10]:
BASE_URL = "https://parking.2759359719sw.workers.dev"

In [11]:
def take_picture(xcoor, ycoor): #basically makes a pipeline: connect to adb, press cam, wait for new file, put in local folder

    adb_path = r"C:\Platform_Tools\platform-tools\adb.exe"    
    adb_ip = "127.0.0.1:5555"                             
    blue_img_folder = "/sdcard/DCIM/LWProPhotos"            
    local = "Drone_Pics"                             

    # connect adb
    subprocess.run([adb_path, "connect", adb_ip], capture_output=True, text=True) # shell: adb.exe connect adb_ip

    # finds a list of the pics before new img capture so that old ones not reused 
    result = subprocess.run(
        [adb_path, "-s", adb_ip, "shell", "ls", blue_img_folder], #shell:C:adb_path -s adb_ip shell ls blue_img_folder
        capture_output=True,
        text=True
    )  #this will spit img file numbers horizontally

    files_before = result.stdout.splitlines()    #puts the img file numbers vertically

    #press app camera
    x = xcoor["take_picture"]
    y = ycoor["take_picture"]
    pyautogui.click(x, y)

    time.sleep(2)

    start_time = time.time()
    new_file = None

    # checks for new files 
    while time.time() - start_time < 10:  #gives a ten second period for new img upload (reduce for faster response later)

        result = subprocess.run(
            [adb_path, "-s", adb_ip, "shell", "ls", blue_img_folder],
            capture_output=True,
            text=True
        )

        files_after = result.stdout.splitlines()  # spits all img names vertically agian

        for i in files_after:   #checks each line with img name 
            if i not in files_before:
                new_file = i   # if string doesnt match, it assumes new img file
                break

        if new_file:  # leave while loop when new file exists
            break

        time.sleep(0.5)

    if new_file is None:
        return None

    # puts where new file would be redirected to
    new_img_path = os.path.join(local, new_file)

    subprocess.run([    #actually copies new img file to local folder 
        adb_path,
        "-s",
        adb_ip,
        "pull",
        blue_img_folder + "/" + new_file,
        new_img_path
    ], capture_output=True, text=True)

    return new_img_path

In [12]:

def get_lots():
    url = f"{BASE_URL}/api/lot"
    response = requests.get(url)
    return response.json()

def get_spaces():
    url = f"{BASE_URL}/api/space"
    response = requests.get(url)
    return response.json()

In [13]:
# function used to upload pictures
def upload_frame(new_img_path, BASE_URL):
  with open(new_img_path, 'rb') as f:
    response = requests.post(f"{BASE_URL}/api/upload-frame", data=f.read(), params={'filename': new_img_path.split('/')[-1]})
    return response.json()


In [ ]:
#main drone control function 
checkpoints = [
    (26.4382271, -80.1050785),  ##first checkpoint   this has an index of 0 for reference 
    (26.4381520, -80.1049876),   ##second checkpoint  this has an index of 1 for reference, we can add more for the future too 
    (26.4382271, -80.1050785)   #third checkpoint goes back to first
]
 
    #(26.46384, -80.21771),  ##park coordinates
    ##(26.46386, -80.21806),   
    ##(26.46384, -80.21771)
current_cp = 0   ##start at the first checkpoint 

lat_thresh = 0.00001   ## based on some measurements done on google earth
lon_thresh = 0.000005    ## based on some measurements done on google earth

# Unlock button
time.sleep(5)
pyautogui.mouseDown(xcoor['unlock_lift'], ycoor['unlock_lift'], button='left')
pyautogui.moveTo(xcoor['unlock_lift'], ycoor['unlock_lift'], duration=1);
pyautogui.mouseUp(xcoor['unlock_lift'], ycoor['unlock_lift'], button='left')
time.sleep(1)

# lift off and slider
time.sleep(1)
pyautogui.click(xcoor['lift_off'], ycoor['lift_off'])
pyautogui.mouseDown(xcoor['lift_slide1'], ycoor['lift_slide1'], button='left')
pyautogui.moveTo(xcoor['lift_slide2'], ycoor['lift_slide2'], duration=1);
pyautogui.mouseUp(xcoor['lift_slide2'], ycoor['lift_slide2'], button='left')
time.sleep(1)


# Rockers
time.sleep(1)
pyautogui.click(xcoor['activate_rocker'], ycoor['activate_rocker'])
time.sleep(1)


while True:
    
    latitude_read, longitude_read = read_coords_stable(n=3, delay=0.04)
    
    if latitude_read is None or longitude_read is None:
        time.sleep(0.1)
        continue

    cp_lat, cp_lon = checkpoints[current_cp]  ##the checkpoint lats and longs are upodated based on the list 


    if abs(latitude_read - cp_lat) <= lat_thresh and abs(longitude_read - cp_lon) <= lon_thresh:  # for whichever checkpoint since the logic is the same, if the thresholds are met go to the next one
        
        new_image_path = take_picture(save_screenshot=True)
        if new_image_path:   #if statement to check if upload happened so no crash      ##!! COMMENT BEFORE SERVER UPLOAD
            upload_frame(new_image_path, BASE_URL)   #send to server                    ##!! COMMENT BEFORE SERVER UPLOAD
            
        current_cp += 1  ##goes to next index or checkpoint on list for next coordinates
        time.sleep(1)

        if current_cp >= len(checkpoints):  ## once the number of elemetns are exceeded on the checkpoint list, land the drone and break
            # Land
            time.sleep(1)
            pyautogui.click(xcoor['lift_off'], ycoor['lift_off'])
            pyautogui.mouseDown(xcoor['lift_slide1'], ycoor['lift_slide1'], button='left')
            pyautogui.moveTo(xcoor['lift_slide2'], ycoor['lift_slide2'], duration=1);
            pyautogui.mouseUp(xcoor['lift_slide2'], ycoor['lift_slide2'], button='left')
            time.sleep(1)
            break

        time.sleep(1)   ##small delay and continue code if more checkpoints available
        continue

    if latitude_read > cp_lat + lat_thresh:     ## the code always adjusts the position from IP to CP1 or CP1 to CP2
        #move drone south
        move_drone('backward',0.2);

    elif latitude_read < cp_lat - lat_thresh:
        #move drone north
        move_drone('forward',0.2);

    if longitude_read > cp_lon + lon_thresh:
        #move drone west
        move_drone('left',0.2);

    elif longitude_read < cp_lon - lon_thresh:
        #move drone east
        move_drone('right',0.2);
        
    time.sleep(0.2)


